# Detecting gravitational waves

In [ ]:
import pycbc.waveform
import pycbc
import matplotlib.pyplot as plt
import numpy as np
import lal as _lal

This notebook gives an initial overview of what SNR is, and pointers on how to compute it.

Initially we can define a few constants that we will use later on.

In [ ]:
sampling_frequency = 1024
duration = 1
dt = 1./sampling_frequency # separation of samples in seconds

We can start by using pycbc to generate the timeseries of a waveform.

In [ ]:
hplus, hcross = pycbc.waveform.get_td_waveform(mass1=36,mass2=30,distance=500,approximant='IMRPhenomPv2',f_lower=30, delta_t=dt)
print('PyCBC waveform has length ',len(hplus))

Define some initial variables

In [ ]:
psd_mag = 1.0e-46#(2e-23)**2               # set a fixed psd magnitude
Nt = sampling_frequency*duration   # define length of time series
df = 1./duration                   # frequency separation

In [ ]:
# create some noise based on psd 
merger_index = np.argmin(np.abs(hplus.sample_times)) # find the sample index of the merger
times_window = hplus.sample_times[merger_index - int(0.75*Nt) : merger_index + int(0.25*Nt)] # select window of times around filter
filter = hplus[int(merger_index - 0.75*Nt) : int(merger_index + 0.25*Nt)] # extract 1s of the waveform surrounding the merger

In [ ]:
with open('./data/DET/DET_white_data_ts.txt', "r") as f:
    times, data = np.loadtxt(f).T

In [ ]:
fig, ax = plt.subplots()
ax.plot(times_window, data, label = "noisy data")
ax.plot(times_window, filter, label = "signal only")
ax.set_ylabel("strain")
ax.set_xlabel("N_samples (time)")
ax.legend()

Compute the matched filter SNR we can use
$$ (s | h) = 4 {\rm Re} \int_{0}^{\infty} \frac{\tilde{s}(f) \tilde{h}^*(f)}{S_{n}(f)}df $$
The matched filter is then defined as 
$$ \rho = \frac{(s|h)}{\sqrt{(h|h)}} $$
Where $\tilde{s}(f)$ is the Fourier transformed detector data, $\tilde{h}^*(f)$ is the complex conjugate of the fourier transformed waveform and $S_{n}(f)$ is the noise power spectral density of the detector.

Below I have started to code up the computation of the SNR. **Complete the function below to estimate the SNR of the signal you loaded in.**

In [ ]:
def compute_snr(filter, data, psd, dt):
    # define come constants
    Nt = len(filter)
    data_duration = Nt*dt

    # compute data ffts (i.e. get frequency domain of data)
    detector_fft = np.fft.rfft(data) * dt
    waveform_fft = np.fft.rfft(filter) * dt

    #compute inner product of signal and data
    s_inner_h =  4/data_duration * np.sum(np.conj(waveform_fft) * detector_fft / psd)

    #
    # rest goes here

    return snr

We can also find the SNR timeseries using the fourier transform trick, this is equivalent to computing the above definition of SNR at every time step (i.e. shifting the template byt one timestep each time)
$$ \rho(t) = \frac{4}{\sqrt{(h|h)}} {\rm Re} \int_{0}^{\infty} \frac{\tilde{s}(f) \tilde{h}^*(f)}{S_{n}(f)} e^{2\pi i f t}df$$

**Try coding up a separate function that computes the SNR timeseries. Does it agree with what you can see in the figure above?**

# Signal detection mock data challenge

## Part 1: detecting a known gravitational wave event

In the data/DET folder there are files named: 
 - **DET2_white_noise_ts.txt** which contains the times and strains for a chunk of empty (noise-only) gravitational wave detector data (this can be used to compute the PSD more accurately than by using a chunk of data which contains a signal).
 - **DET2_white_data_ts.txt** which contains the times and strains for a chunk of gravitational wave detector data which includes a single gravitational-wave signal for a $m_1 = 35$, $m_2 = 30$ solar mass binary black hole.
   

**Load the data and try and find the signal within the second file using the function you created above.** What time does it occur? This signal is much weaker than the example above so think about how one can say the SNR is statistically significant.

## Part 2: creating a template bank

If you already know the masses of your black holes, Part 1 was hopefully quite straightforward. But what if you *didn't* know the masses? What if, for example, you just knew that black holes had masses somewhere in the range of 20 to 40 solar masses?

**Develop a search which can identify the time of the GW event without assuming prior knowledge of its masses.** How many templates do you need? How will you ensure that you cover the required parameter space? Test it on the data above. What do you learn about the masses of the black hole from this approach?

## Part 3: hunting for unknown signals

In the data folder there is a file named timeseries_data_multisignal.txt which contains a 100 second chunk of gravitational wave data which contains *at least* 5 gravitational wave detections (and possibly many more...). **Use your search algorithm to hunt for signals in this stretch of data.** How many can you find, what times do they occur, and what are their masses (approximately)? Check your answers with a demonstrator!

Note: the H1 and L1 columns in this file correspond to the data from the LIGO Hanford and LIGO Livingston detectors. **Test your your algorithm using just one dataset to start!**

## Part 4: Now it's up to you 

Once you've got this far, congratulations! You now know the basics of gravitational wave detection and how to construct a template bank. Use the rest of your time in the labs to investigate an aspect of gravitational wave detection more deeply. For example, you might like to:

- Try incorporating data from a second gravitational wave detector into the mock data challenge you just completed. How do your results change? Perhaps there will be even more signals to be discovered...
- Try out your search algorithm on a real stretch of gravitational wave data!
- Perhaps you have some other ideas of your own...?

Speak to a demonstrator to discuss ideas.